# Can an LLM write training data for a BERT model?

### Gwilym Hughes, Summer 2026 Internship

The main reason for looking towards synthetic training data to train AI models is that annotation is the bottleneck to starting new projects. Getting an LLM to write labelled examples instead skips the bottleneck. In theory the knowledge a large LLM has in it's weights should be sufficient to generate good training data for a smaller model like BERT, ie. distillation. This notebook is the result of a 8-week internship trying this for drug-drug interaction extraction.

There are two ways to spend an LLM budget, either have it label text that already exists, or have it write new labelled text. We ran both, on the same sentences with the same entity spans, so the only thing varying is where the text came from and where the labels came from.

| what the model trains on | dev micro-F1 |
|---|---|
| real text, human labels | 0.785 |
| real text, LLM labels | 0.657 |
| LLM text, LLM labels | 0.462 |

Swapping human labels for LLM labels costs 0.128, and swapping real text for generated text costs another 0.195. So the text hurts more than the labels do, which is not what we expected, because the synthetic labels are correct by construction, since the generator was told what relation to write.

The rest of this notebook rebuilds that result from scratch and then digs into why the generated text is so much worse.

Set `FAST = True` to run a quick version of the notebook, though it will be more noisy.

## Setup

You need a GPU with ~16GB free, a copy of DDI-2013, and an OpenAI-compatible endpoint.
We used gpt-oss-120b, so mileage may vary with other models. The endpoint must support the chat API, and you need an API key.


In [ ]:
!pip install -q transformers torch scikit-learn openai tqdm pandas

In [ ]:
import os, re, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

FAST = True

SEEDS = [0] if FAST else [0, 1, 2]
MAX_SENTENCES = 600 if FAST else None
EPOCHS = 3

CORPUS = Path(os.environ.get("DDI_CORPUS", "../DDICorpusBrat"))
MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

API_BASE = os.environ.get("LLM_API_BASE", "http://api.llm.apps.os.dcs.gla.ac.uk/v1")
API_KEY = os.environ.get("IDA_LLM_API_KEY", "EMPTY")
LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-oss-120b")

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print(f"device {DEVICE}, fast={FAST}")

## Getting the corpus

DDI-2013 came out of SemEval 2013 Task 9. It isn't redistributable here, so the brat-format release has to be fetched separately, from the [DDICorpus mirror](https://github.com/isegura/DDICorpus) or the task page.

This is the step most likely to go wrong, and it went wrong for us. Different distributions use different folder layouts. So rather than hardcode a path, the function below goes looking for wherever the `.ann` files actually are.

In [ ]:
def find_ann_dirs(root):
    if not root.exists():
        raise FileNotFoundError(
            f"{root} doesn't exist. Download DDI-2013 (brat format) and put it there, "
            f"or set DDI_CORPUS to point at it.")

    dirs = sorted({p.parent for p in root.rglob("*.ann")})
    if not dirs:
        raise FileNotFoundError(
            f"No .ann files under {root}. The XML release won't work here, it has to be "
            f"the brat one.")

    splits = {"train": [], "test": []}
    for d in dirs:
        rel = str(d.relative_to(root)).lower()
        splits["test" if "test" in rel else "train"].append(d)

    for name, ds in splits.items():
        n = sum(len(list(d.glob("*.ann"))) for d in ds)
        print(f"{name}: {n} docs in {len(ds)} directories")
        for d in ds:
            print(f"    {d.relative_to(root)}")
    return splits

SPLITS = find_ann_dirs(CORPUS)

## What one training instance looks like

Worth slowing down here, because this is the part that makes the task harder than it looks and it's what the generator kept getting wrong.

The label doesn't belong to the sentence. It belongs to an ordered *pair* of drug mentions inside the sentence. A sentence with four drugs gives six pairs, and those six pairs can carry six different labels. Five labels are available: MECHANISM (a pharmacokinetic change), EFFECT (a clinical or pharmacodynamic consequence), ADVISE (a recommendation about co-administration), INT (an interaction with no further detail), and NONE.

Parsing the corpus first, then a look at a single sentence.

In [ ]:
from dataclasses import dataclass, field

LABELS = ["NONE", "MECHANISM", "EFFECT", "ADVISE", "INT"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
POSITIVE = [LABEL2ID[l] for l in LABELS if l != "NONE"]

@dataclass
class Entity:
    eid: str; text: str; start: int; end: int

@dataclass
class Sentence:
    doc_id: str; text: str
    entities: list = field(default_factory=list)
    relations: dict = field(default_factory=dict)

def parse_doc(ann_path):
    txt = ann_path.with_suffix(".txt")
    if not txt.exists():
        return None
    text = txt.read_text(encoding="utf-8", errors="replace")

    ents, rels = {}, {}
    for line in ann_path.read_text(encoding="utf-8", errors="replace").splitlines():
        parts = line.rstrip().split("\t")
        if len(parts) < 2 or not parts[0]:
            continue
        if parts[0][0] == "T" and len(parts) >= 3:
            _, span = parts[1].split(" ", 1)
            offs = [int(x) for c in span.split(";") for x in c.split()]
            ents[parts[0]] = Entity(parts[0], parts[2], min(offs), max(offs))
        elif parts[0][0] in "RE":
            f = parts[1].split()
            args = [a.split(":", 1)[1] for a in f[1:] if ":" in a]
            if len(args) == 2:
                rels[(args[0], args[1])] = f[0]

    return Sentence(ann_path.stem, text, list(ents.values()), rels)

def load(dirs):
    out = []
    for d in dirs:
        for ann in sorted(d.glob("*.ann")):
            s = parse_doc(ann)
            if s and len(s.entities) >= 2:
                out.append(s)
    return out

train_docs, test_docs = load(SPLITS["train"]), load(SPLITS["test"])
print(f"{len(train_docs)} train sentences, {len(test_docs)} test sentences")

Here's one sentence with a few drugs in it, and every pair it produces.

In [ ]:
ex = next(s for s in train_docs if 3 <= len(s.entities) <= 5 and s.relations)
print(ex.text.strip(), "\n")

rows = []
for i, a in enumerate(ex.entities):
    for b in ex.entities[i+1:]:
        lab = ex.relations.get((a.eid, b.eid)) or ex.relations.get((b.eid, a.eid)) or "NONE"
        rows.append({"drug A": a.text, "drug B": b.text, "label": lab})
pd.DataFrame(rows)

Most of those are NONE. About 84% of all pairs in the corpus are, and that's before the guideline conventions, which are where most of the difficulty lives. Two that matter later.

Two mentions of the same drug, or a drug and a group containing it, are NONE no matter what the sentence says. And when a sentence supports more than one label for a pair, there's a fixed priority order rather than a judgement call.

Neither of those is pharmacology. They're annotation convention, and a model that knows the drugs perfectly still gets them wrong until it's told the rules.

## Turning a pair into something a classifier can eat

The two candidate entities get marked in place and every other drug mention is blinded, so the model can't cheat off drug identity.

This function is load-bearing and gets things wrong silently. A rewrite of ours corrupted the human baseline and it took a while to notice, because nothing throws, the numbers just move. Any change to it should be followed by re-running the human baseline and checking it still lands near 0.79.

In [ ]:
def render_pair(sent, a, b):
    marks = []
    for e in sent.entities:
        tok = "@DRUG-A$" if e.eid == a.eid else "@DRUG-B$" if e.eid == b.eid else "@DRUG-OTHER$"
        marks.append((e.start, e.end, tok))

    out, cur = [], 0
    for start, end, tok in sorted(marks):
        if start < cur:
            continue
        out.append(sent.text[cur:start]); out.append(tok); cur = end
    out.append(sent.text[cur:])
    return " ".join("".join(out).split())

def to_instances(docs, limit=None):
    docs = docs[:limit] if limit else docs
    rows = []
    for s in docs:
        ents = sorted(s.entities, key=lambda e: e.start)
        for i, a in enumerate(ents):
            for b in ents[i+1:]:
                lab = s.relations.get((a.eid, b.eid)) or s.relations.get((b.eid, a.eid)) or "NONE"
                if lab in LABEL2ID:
                    rows.append({"doc_id": s.doc_id, "text": render_pair(s, a, b),
                                 "label": LABEL2ID[lab], "n_entities": len(ents)})
    return pd.DataFrame(rows)

train_df = to_instances(train_docs, MAX_SENTENCES)
test_df = to_instances(test_docs)
print(train_df.iloc[0]["text"][:250])

### One caveat before any numbers

Most published DDI-2013 papers apply negative instance filtering first, a rule-based deletion of pairs that are trivially NONE, like same-drug pairs and drugs sitting in a coordinated list. We don't. That is a large part of why published micro-F1 on this benchmark runs anywhere from 65 to 90, and much of that spread is preprocessing rather than modelling.

So the 0.79 human baseline here isn't comparable to the numbers in those papers. It's measured on a harder instance set. Everything in this notebook is compared against that same baseline, so the internal comparisons hold, but 0.79 should not be read against anyone else's 0.90.

## The classifier

BiomedBERT, five-way, micro-F1 over the four positive classes. NONE is excluded from the metric, since including it makes everything look good: a model predicting NONE for every pair scores 0.84.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.metrics import f1_score, precision_score, recall_score

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class Pairs(Dataset):
    def __init__(self, df, max_len=192):
        self.enc = tokenizer(list(df["text"]), truncation=True,
                             max_length=max_len, padding="max_length")
        self.y = list(df["label"])
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        d = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        d["labels"] = torch.tensor(self.y[i]); return d

def train_and_eval(train_df, eval_df, seed=0, epochs=EPOCHS, bs=32, lr=3e-5):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS)).to(DEVICE)
    dl = DataLoader(Pairs(train_df), batch_size=bs, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.1*len(dl)*epochs), len(dl)*epochs)

    model.train()
    for _ in range(epochs):
        for batch in dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch); out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()

    model.eval(); preds = []
    with torch.no_grad():
        for batch in DataLoader(Pairs(eval_df), batch_size=64):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(**{k: v for k, v in batch.items() if k != "labels"}).logits
            preds += logits.argmax(-1).cpu().tolist()

    y = list(eval_df["label"])
    r = {"f1": f1_score(y, preds, labels=POSITIVE, average="micro", zero_division=0),
         "p": precision_score(y, preds, labels=POSITIVE, average="micro", zero_division=0),
         "r": recall_score(y, preds, labels=POSITIVE, average="micro", zero_division=0)}
    del model; torch.cuda.empty_cache()
    return r

def run_arm(name, tr, ev, seeds=SEEDS):
    scores = []
    for s in seeds:
        m = train_and_eval(tr, ev, seed=s)
        scores.append(m["f1"])
        print(f"  {name} seed {s}: F1 {m['f1']:.3f}  P {m['p']:.3f}  R {m['r']:.3f}")
    return {"arm": name, "f1": float(np.mean(scores)),
            "sd": float(np.std(scores)), "n": len(tr)}

results = [run_arm("human", train_df, test_df)]
pd.DataFrame(results)

## Arm two: real text, LLM labels

Same sentences, same spans, same pairs. The gold labels get thrown away and an LLM is asked for them instead.

The prompt spells out the guideline conventions. That isn't decoration. Without them the annotator's NONE recall sits around 0.46, and with them it's about 0.92. It's the single biggest lever in this arm, which fits the point above about the conventions being where the difficulty is.

In [ ]:
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

client = OpenAI(base_url=API_BASE, api_key=API_KEY)

ANNOTATOR = """You annotate drug-drug interactions following the DDI-Extraction 2013 guidelines.

For each pair you are asked about, give exactly one label.

  MECHANISM  a pharmacokinetic interaction: one drug changes the absorption, metabolism,
             clearance, plasma levels or exposure of the other.
  EFFECT     a pharmacodynamic or clinical outcome of the two together.
  ADVISE     a recommendation about using the two together.
  INT        the text says only that they interact, with no detail.
  NONE       everything else.

Rules that decide the hard cases:

If a pharmacokinetic change is asserted, the label is MECHANISM even when a clinical
consequence is also mentioned. If advice is given and an effect is also stated, ADVISE.

Both drugs must participate in the claim. If the sentence asserts something about one and
merely mentions the other, the pair is NONE.

A denial is NONE. "No significant change in the pharmacokinetics of X was seen with Y"
is NONE. A statement that something has not been studied is also NONE.

If the same drug is named twice, only the mention that participates counts.

Answer with JSON only: {"labels": [{"id": <int>, "label": "<LABEL>"}, ...]}"""

def annotate(chunk):
    user = "\n".join(f'{r["id"]}. {r["text"]}' for r in chunk)
    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL, temperature=0.0,
            messages=[{"role": "system", "content": ANNOTATOR},
                      {"role": "user", "content": user}])
        raw = re.sub(r"^```(?:json)?|```$", "", resp.choices[0].message.content.strip(), flags=re.M)
        got = {int(d["id"]): d["label"] for d in json.loads(raw)["labels"]}
    except Exception:
        got = {}
    return [(r["id"], LABEL2ID.get(got.get(r["id"], "NONE"), 0)) for r in chunk]

def llm_annotate(df, size=8, workers=32):
    recs = [{"id": i, "text": t} for i, t in enumerate(df["text"])]
    chunks = [recs[i:i+size] for i in range(0, len(recs), size)]
    out = {}
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for res in tqdm(ex.map(annotate, chunks), total=len(chunks)):
            out.update(dict(res))
    d = df.copy().reset_index(drop=True)
    d["label"] = [out.get(i, 0) for i in range(len(d))]
    return d

llm_df = llm_annotate(train_df)

Before training on those, compare them to the gold labels directly. The overall agreement is less interesting than its shape.

In [ ]:
y, yhat = list(train_df["label"]), list(llm_df["label"])
print(f"positive precision {precision_score(y, yhat, labels=POSITIVE, average='micro', zero_division=0):.3f}")
print(f"positive recall    {recall_score(y, yhat, labels=POSITIVE, average='micro', zero_division=0):.3f}")
print(f"human positives {sum(1 for v in y if v)}, LLM positives {sum(1 for v in yhat if v)}")

Recall comes out at or above human. Precision is well below, and the model predicts roughly 40% more positives than there are. The deficit isn't a failure to understand the pharmacology, it's that co-mention gets read as interaction. That points at a fixable prompt problem rather than a ceiling.

In [ ]:
results.append(run_arm("llm-labels", llm_df, test_df))
pd.DataFrame(results)

## Arm three: generate the text too

Now the sentences are written as well as labelled. The generator is told which pair to make positive and with which label, so the labels are correct by construction. If label noise were the whole story, this arm ought to beat the previous one.

It doesn't, and the two constraints in the prompt below were both added after watching earlier versions fail.

The generator has to include drugs that aren't in the relation. Without them every sentence has exactly one pair and that pair is positive, so the classifier learns that one pair means positive and falls over on real multi-drug sentences. There's a probe for this further down.

And it mustn't flag those non-participants. Early versions marked them with phrases like "given earlier" or "as a comparator", which moves the shortcut rather than removing it: the model then learns that role language nearby means NONE.

In [ ]:
GENERATOR = """You write single sentences that could come from a drug package insert or a
biomedical abstract.

You get a target relation label, two participant drugs, and some non-participant drugs.

Write exactly one sentence containing every drug named, spelled exactly as given. The two
participants must stand in the specified relation. The non-participants must appear as
ordinary co-mentions with nothing asserted between them and anything else. Don't mark
them out with phrases like "given earlier", "as a comparator" or "unlike". They should
read as unremarkable.

Vary the sentence structure. Don't reuse a template. Don't state the label.

  MECHANISM  pharmacokinetic mechanism, naming the process.
  EFFECT     pharmacodynamic effect or clinical consequence.
  ADVISE     recommendation about co-administration.
  INT        an interaction occurs, nothing further said.

Answer with JSON only: {"sentence": "..."}"""

DRUGS = sorted({e.text for s in train_docs for e in s.entities if len(e.text) > 3})

def generate(spec):
    label, a, b, others = spec
    user = (f"label: {label}\nparticipants: {a}, {b}\n"
            f"non-participants: {', '.join(others) if others else '(none)'}")
    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL, temperature=1.0,
            messages=[{"role": "system", "content": GENERATOR},
                      {"role": "user", "content": user}])
        raw = re.sub(r"^```(?:json)?|```$", "", resp.choices[0].message.content.strip(), flags=re.M)
        sent = json.loads(raw)["sentence"].strip()
    except Exception:
        return None
    for d in [a, b] + others:
        if d.lower() not in sent.lower():
            return None          # a drug name didn't survive generation
    return {"sentence": sent, "label": label, "a": a, "b": b, "others": others}

rng = random.Random(0)
specs = []
for _ in range(300 if FAST else 2400):
    label = rng.choice(["MECHANISM", "EFFECT", "ADVISE", "INT"])
    picked = rng.sample(DRUGS, 2 + rng.choice([0, 1, 1, 2, 2, 3]))
    specs.append((label, picked[0], picked[1], picked[2:]))

with ThreadPoolExecutor(max_workers=32) as ex:
    generated = [g for g in tqdm(ex.map(generate, specs), total=len(specs)) if g]

print(f"{len(generated)}/{len(specs)} kept\n")
print(generated[0]["sentence"])

Those sentences have to become pair instances the same way the real ones did, including the NONE pairs. Those NONE pairs are the whole reason the non-participants are there.

In [ ]:
def synth_to_instances(gen):
    rows = []
    for i, g in enumerate(gen):
        found = []
        for n in [g["a"], g["b"]] + g["others"]:
            m = re.search(re.escape(n), g["sentence"], re.I)
            if m:
                found.append(Entity(n, n, m.start(), m.end()))
        if len(found) < 2:
            continue
        s = Sentence(f"synth{i}", g["sentence"], found, {(g["a"], g["b"]): g["label"]})
        ents = sorted(found, key=lambda e: e.start)
        for j, x in enumerate(ents):
            for y in ents[j+1:]:
                lab = s.relations.get((x.eid, y.eid)) or s.relations.get((y.eid, x.eid)) or "NONE"
                rows.append({"doc_id": s.doc_id, "text": render_pair(s, x, y),
                             "label": LABEL2ID[lab], "n_entities": len(ents)})
    return pd.DataFrame(rows)

synth_df = synth_to_instances(generated)
print(f"{len(synth_df)} instances, {(synth_df['label'] != 0).mean():.1%} positive "
      f"(corpus is {(train_df['label'] != 0).mean():.1%})")

results.append(run_arm("synthetic", synth_df, test_df))
pd.DataFrame(results)

## Why is the generated text worse?

Downstream F1 says a dataset is bad. It doesn't say what's wrong with it. These three probes do, they're cheap, and each of them caught something that would otherwise have been missed.

### Probe 1: does pair count predict the label?

Group instances by how many drugs the sentence contains and look at the positive rate. On the real corpus it's fairly flat. On our first generator it wasn't. One-pair sentences ran about 75% positive and multi-pair sentences close to 0%, so the classifier could get most of the way by counting drugs.

In [ ]:
def composition(df, name):
    g = df.groupby("n_entities")["label"].agg(pos_rate=lambda s: (s != 0).mean(), n="size")
    g.insert(0, "dataset", name); return g

pd.concat([composition(train_df, "human"), composition(synth_df, "synthetic")])

### Probe 2: train on the structure with the words deleted

Strip out everything except the entity markers and train on that. Anything above chance is signal that leaked into structure rather than content.

There's a trap here that cost us a fortnight of misplaced confidence. This has to be calibrated against a size-matched human sample, not against a threshold picked by reasoning. Full human train scores about 0.152 on this probe with sd 0.020, but 1.6k-instance subsamples of the same data range from 0.00 to 0.17 with sd 0.085. We had a battery of four such gates, all with thresholds we'd argued our way to, and all four passed on a dataset that then scored 0.379. A training run takes twenty minutes, so it's usually quicker to just train.

In [ ]:
def strip_words(df):
    d = df.copy()
    d["text"] = d["text"].str.replace(r"[^@\w$-]+", " ", regex=True)
    d["text"] = d["text"].apply(lambda t: " ".join(w for w in t.split() if w[0] == "@"))
    return d

n = min(len(train_df), len(synth_df))
for name, df in [("human", train_df), ("synthetic", synth_df)]:
    m = train_and_eval(strip_words(df.sample(n, random_state=0)),
                       strip_words(test_df.sample(min(2000, len(test_df)), random_state=0)),
                       seed=SEEDS[0], epochs=2)
    print(f"{name}: partial-input F1 {m['f1']:.3f} at n={n}")

### Probe 3: is the label a function of the construction?

This one caught the most interesting failure of the project. A later generator drew from an inventory of 20 guideline-grounded sentence frames, which sounds like an improvement and scored like one. But normalised mutual information between frame and label came out at exactly 1.000, and a bag of words could name the frame from the masked sentence with 0.961 accuracy over 19 classes.

The label was therefore fully determined by which frame got used, and the classifier had nothing to learn but the frames. Real sentences don't carry frame signatures, so at test time there was nothing to fall back on. Separating the assertion that carries the label from modifiers that don't was worth +0.095 F1, entirely in recall.

The cheap version of the check is below: mask the drug names, then see whether a linear model can recover the label from bag-of-words.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

mask = lambda t: re.sub(r"@DRUG-[AB]\$|@DRUG-OTHER\$", "@DRUG$", t)

for name, df in [("human", train_df), ("synthetic", synth_df)]:
    pos = df[df["label"] != 0]
    if len(pos) < 50:
        continue
    X = CountVectorizer(ngram_range=(1, 2), min_df=2).fit_transform(pos["text"].map(mask))
    acc = cross_val_score(LogisticRegression(max_iter=2000), X, pos["label"], cv=3).mean()
    print(f"{name}: label recoverable at {acc:.3f} (chance {1/pos['label'].nunique():.3f})")

## Results

With `FAST = True` these are one seed on a fraction of the data, so they come out low. The ordering is the thing to read.

In [ ]:
tab = pd.DataFrame(results)
tab["vs human"] = tab["f1"] - tab.loc[tab["arm"] == "human", "f1"].iloc[0]
tab

## What we would do differently

Run the head-to-head first. A literature review in week 6 pointed fairly clearly at labelling beating generation. We ran the comparison in week 8, after six weeks of generator iteration that the comparison made largely beside the point. It costs one afternoon.

Expect shortcuts to arrive one at a time. Removing the pair-count shortcut improved precision and did nothing for recall, which looked like the fix having failed. It hadn't. A second shortcut was now the binding constraint. Each fix reveals the next one, so a flat result after a correct change is information rather than a dead end.

Distributional similarity metrics didn't tell us what we wanted to know. We ran ten of them, and none predicted either of the two effects that mattered. There's a live argument in the literature about whether diversity metrics specifically are the exception, and we never tested that properly, so this is one data point rather than a verdict.

Watch which way the filters are biased. Twice, a filter's selection effect happened to point in the same direction as the hypothesis being tested, and both times it was caught by reading raw output rather than aggregate numbers.

## Where to take it next

Negative instance filtering, so the numbers are comparable to published work. Ten-seed replication, since three isn't enough for effects around 0.04. And adjudicating the apparent gold errors: the annotator's false positives include cases where it looks right and the corpus looks wrong, and nobody has checked.